<div style="background:linear-gradient(135deg,#143840 0%,#2B6264 100%);border-radius:14px;padding:32px 36px;color:#fff;font-family:'DM Sans',Arial,sans-serif;">
  <div style="font-size:11px;letter-spacing:2px;text-transform:uppercase;color:#FF4B31;font-weight:700;margin-bottom:10px;">Solutions Onboarding &middot; IBOR Training</div>
  <div style="font-size:30px;font-weight:700;line-height:1.15;margin-bottom:10px;">IBOR NB04 &mdash; Holdings & Position Management</div>
  <div style="font-size:15px;color:rgba(255,255,255,.82);max-width:640px;line-height:1.55;">Query holdings across all portfolios, with bitemporal queries, sub-holding-key breakdowns and position drill-down.</div>
</div>

<sub>IBOR pack sequence: NB00 &nbsp;&rarr;&nbsp; NB01 &nbsp;&rarr;&nbsp; NB02 &nbsp;&rarr;&nbsp; NB03 &nbsp;&rarr;&nbsp; **NB04** &nbsp;&rarr;&nbsp; NB05 &nbsp;&rarr;&nbsp; NB06 &nbsp;&rarr;&nbsp; NB07 &nbsp;&rarr;&nbsp; NB08</sub>

# NB04: Holdings & Position Management

Queries holdings across all portfolios, demonstrates bitemporal queries, SHK breakdowns, and position drill down.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lusid-sdk', 'lumipy', '-q'],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
import os, json
from datetime import datetime, timedelta, timezone
import pandas as pd
import lusid as lu
import lusid.models as lm
import lumipy as lp
pd.set_option("display.max_columns", None)
SCOPE = 'ibor-training-v9'
QUOTE_SCOPE = 'ibor-training-v9-quotes'

def get_factory(app='lusid'):
    with open("secrets.json") as f: secrets = json.load(f)
    pat = secrets.get("api", {}).get("accessToken")
    module = __import__(app)
    if pat:
        return module.extensions.SyncApiClientFactory(config_loaders=[
            module.extensions.ArgsConfigurationLoader(api_url=secrets["api"]["lusidUrl"], access_token=pat)])
    return module.extensions.SyncApiClientFactory(config_loaders=[
        module.extensions.SecretsFileConfigurationLoader("secrets.json")])

def get_lumi():
    with open("secrets.json") as f: secrets = json.load(f)
    pat = secrets.get("api", {}).get("accessToken")
    if pat:
        return lp.get_client(access_token=pat, api_url=secrets["api"]["lusidUrl"].replace("/api", "/honeycomb"))
    return lp.get_client(api_secrets_file="secrets.json")

lusid_factory = get_factory()
lumi = get_lumi()
print('Ready')
transaction_portfolios_api = lusid_factory.build(lu.TransactionPortfoliosApi)

---
## Holdings Across All Portfolios

In [ ]:
query = """SELECT PortfolioCode, LusidInstrumentId, Units, CostAmount, HoldingType, SubHoldingKey
FROM Lusid.Portfolio.Holding
WHERE PortfolioScope = '{SCOPE}' AND EffectiveAt = '2024-09-30'
ORDER BY PortfolioCode, Units DESC
LIMIT 50"""
query = query.replace('{SCOPE}', SCOPE)
df = lumi.run(query, quiet=True)
print(f"Total holdings across all portfolios: {len(df)}")
display(df)

---
## Bitemporal Query: Before and After a Sell

In [ ]:
# IBOR-EQ had sells in August 2024. Check AAPL before and after.
for dt_str in ["2024-07-31", "2024-08-15"]:
    dt = datetime.fromisoformat(dt_str).replace(tzinfo=timezone.utc)
    resp = transaction_portfolios_api.get_holdings(
        scope=SCOPE, code="IBOR-EQ", effective_at=dt.isoformat())
    for h in resp.values:
        if 'AAPL' in str(h.instrument_uid) or 'DQI' in str(h.instrument_uid) or 'DB4' in str(h.instrument_uid):
            print(f"  {dt_str}: {h.instrument_uid} = {h.units:,.0f} units ({h.holding_type})")

---
## SHK Breakdown (Strategy + Custodian)

In [ ]:
query = """SELECT PortfolioCode, LusidInstrumentId, Units, HoldingType, SubHoldingKey
FROM Lusid.Portfolio.Holding
WHERE PortfolioScope = '{SCOPE}' AND PortfolioCode = 'IBOR-FI' AND EffectiveAt = '2024-09-30'
LIMIT 20"""
query = query.replace('{SCOPE}', SCOPE)
pd.set_option('display.max_colwidth', None)
df = lumi.run(query, quiet=True)
print("IBOR-FI Holdings with SHK breakdown:")
for _, row in df.iterrows():
    print(f"  {row['LusidInstrumentId']}: {row['Units']:,.0f} ({row['HoldingType']})")
    print(f"    SHK: {row['SubHoldingKey']}")

---
## Multi Portfolio Summary

In [ ]:
query = """SELECT PortfolioCode, HoldingType, COUNT(*) as Holdings, SUM(CostAmount) as TotalCost
FROM Lusid.Portfolio.Holding
WHERE PortfolioScope = '{SCOPE}' AND EffectiveAt = '2024-09-30'
GROUP BY PortfolioCode, HoldingType
ORDER BY PortfolioCode"""
query = query.replace('{SCOPE}', SCOPE)
df = lumi.run(query, quiet=True)
print("Portfolio summary at 2024-09-30:")
display(df)
print("\nNB04 Complete. Proceed to NB05: Corporate Actions & Lifecycle Events.")